# 03 - Vector Search: Endpoint & Index Setup

This notebook creates the Databricks AI Search infrastructure:
1. **Vector Search Endpoint** - Compute for serving similarity queries
2. **Delta Sync Index** with Databricks-managed embeddings

**Key Design Decisions:**
- **Delta Sync Index** (not Direct Access) - Automatically syncs with source Delta table
- **Databricks-managed embeddings** - Uses `system.ai.qwen3-embedding-0-6b` (1024 dimensions)
- **Triggered sync** - Index updates when pipeline runs (not continuous, to save cost)

> **Prerequisites:** Run `01_data_setup` and the DLT pipeline (`02_ingestion_pipeline`) first.

In [0]:
%run ./00_config

# RAG Chatbot for Project Documentation
## Configuration & Shared Parameters

This notebook defines all shared configuration parameters used across the RAG pipeline.

**Architecture Overview:**
1. **Data Ingestion** (Lakeflow SDP): PDF → Parse → Chunk → Delta Table
2. **Vector Search**: Delta Sync Index with Databricks-managed embeddings
3. **RAG Chain**: Vector Search Retriever + Claude Haiku 4.5 LLM
4. **Serving**: Model Serving Endpoint → Streamlit App on Databricks Apps
5. **Observability**: MLflow Tracing + GenAI Evaluation

✅ Configuration loaded successfully.
   Catalog: ashish_kamboj_rag
   Schema: project_docs_rag
   Volume: /Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents
   Vector Index: ashish_kamboj_rag.project_docs_rag.gold_chunks_index
   Embedding Model: system.ai.qwen3-embedding-0-6b
   LLM Model: system.ai.claude-haiku-4-5


In [0]:
%pip install -U databricks-vectorsearch
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 16.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Not uninstalling requests at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-c32540fa-b8f1-4681-9c96-e1eddf27da0a
    Can't uninstall 'requests'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-c32540fa-b8f1-4681-9c96-e1eddf27da0a
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.8.1
    Not uninstalling mlflow-skinny at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/env

In [0]:
%run ./00_config

# RAG Chatbot for Project Documentation
## Configuration & Shared Parameters

This notebook defines all shared configuration parameters used across the RAG pipeline.

**Architecture Overview:**
1. **Data Ingestion** (Lakeflow SDP): PDF → Parse → Chunk → Delta Table
2. **Vector Search**: Delta Sync Index with Databricks-managed embeddings
3. **RAG Chain**: Vector Search Retriever + Claude Haiku 4.5 LLM
4. **Serving**: Model Serving Endpoint → Streamlit App on Databricks Apps
5. **Observability**: MLflow Tracing + GenAI Evaluation

✅ Configuration loaded successfully.
   Catalog: ashish_kamboj_rag
   Schema: project_docs_rag
   Volume: /Volumes/ashish_kamboj_rag/project_docs_rag/raw_documents
   Vector Index: ashish_kamboj_rag.project_docs_rag.gold_chunks_index
   Embedding Model: system.ai.qwen3-embedding-0-6b
   LLM Model: system.ai.claude-haiku-4-5


In [0]:
# =============================================================================
# CREATE VECTOR SEARCH ENDPOINT
# =============================================================================
# The endpoint provides compute resources for serving similarity search queries.
# One endpoint can host multiple indexes.
# =============================================================================

from databricks.vector_search.client import VectorSearchClient
import time

vsc = VectorSearchClient()

# Check if endpoint already exists
existing_endpoints = [ep["name"] for ep in vsc.list_endpoints().get("endpoints", [])]

if VECTOR_SEARCH_ENDPOINT not in existing_endpoints:
    print(f"⏳ Creating Vector Search endpoint: {VECTOR_SEARCH_ENDPOINT}...")
    vsc.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT,
        endpoint_type="STANDARD"  # STANDARD for production workloads
    )
    
    # Wait for endpoint to be ready
    while True:
        ep = vsc.get_endpoint(VECTOR_SEARCH_ENDPOINT)
        status = ep.get("endpoint_status", {}).get("state", "UNKNOWN")
        if status == "ONLINE":
            print(f"✅ Endpoint '{VECTOR_SEARCH_ENDPOINT}' is ONLINE.")
            break
        elif status in ["PROVISIONING_FAILED", "INTERNAL_ERROR"]:
            raise RuntimeError(f"❌ Endpoint creation failed: {status}")
        print(f"   Status: {status} - waiting...")
        time.sleep(30)
else:
    print(f"✅ Endpoint '{VECTOR_SEARCH_ENDPOINT}' already exists.")

/home/spark-c32540fa-b8f1-4681-9c96-e1/.ipykernel/73/command-8587934572642152-3253875580:8: DeprecationWarning: databricks-vectorsearch is deprecated and has been renamed to databricks-ai-search. Imports under 'databricks.vector_search.*' will continue to work as a thin re-export of 'databricks.ai_search.*', but new code should switch to 'pip install databricks-ai-search' and 'from databricks.ai_search.* import ...'.
  from databricks.vector_search.client import VectorSearchClient


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
⏳ Creating Vector Search endpoint: rag_chatbot_vs_endpoint...
✅ Endpoint 'rag_chatbot_vs_endpoint' is ONLINE.


## ⚠️ Enable Change Data Feed on Pipeline Table

The `gold_validated_chunks` table is created by the SQL DLT pipeline (`02_ingestion_pipeline.sql`). 

**You cannot use `ALTER TABLE` on pipeline-managed tables.** Instead:

1. **Open the pipeline SQL file**: [02_ingestion_pipeline.sql](#file-1050843361480823)
2. **Find the `gold_validated_chunks` definition** (around line 109) and add `TBLPROPERTIES`:
   ```sql
   CREATE OR REFRESH MATERIALIZED VIEW gold_validated_chunks (
     CONSTRAINT valid_chunk_id EXPECT (chunk_id IS NOT NULL) ON VIOLATION DROP ROW,
     CONSTRAINT valid_chunk_text EXPECT (chunk_to_embed IS NOT NULL AND LENGTH(chunk_to_embed) > 50) ON VIOLATION DROP ROW,
     CONSTRAINT valid_source EXPECT (source_file IS NOT NULL) ON VIOLATION DROP ROW
   )
   TBLPROPERTIES (delta.enableChangeDataFeed = true)  -- Add this line
   COMMENT 'Quality-validated chunks for Vector Search indexing'
   AS SELECT * FROM LIVE.gold_document_chunks;
   ```
3. **Save the file** and **run a pipeline update** to apply the change
4. **Then return here** and run cell 7 to create the Vector Search index

> **Note**: Change Data Feed is required for Delta Sync indexes to track table changes.

In [0]:
# =============================================================================
# CREATE DELTA SYNC INDEX WITH DATABRICKS-MANAGED EMBEDDINGS
# =============================================================================
# This creates a Delta Sync Index that:
# - Automatically computes embeddings using the specified model
# - Syncs with the source Delta table (gold_validated_chunks)
# - Uses triggered sync mode (syncs on demand, not continuously)
#
# The index uses:
# - chunk_id as the primary key
# - chunk_to_embed as the source column for embedding computation
# - chunk_to_retrieve and source_file as additional metadata columns
# =============================================================================

# Source table from the DLT pipeline (validated chunks)
source_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.gold_validated_chunks"

# Fix: Use the correct endpoint name (databricks- prefix, not system.ai.)
embedding_model = "databricks-qwen3-embedding-0-6b"

try:
    # Check if index already exists
    existing_index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT,
        index_name=VECTOR_INDEX_NAME
    )
    print(f"✅ Index '{VECTOR_INDEX_NAME}' already exists.")
    print(f"   Status: {existing_index.describe().get('status', {}).get('ready', 'Unknown')}")

except Exception as e:
    if "NOT_FOUND" in str(e) or "RESOURCE_DOES_NOT_EXIST" in str(e) or "NotFound" in str(e) or "does not exist" in str(e).lower():
        print(f"⏳ Creating Delta Sync Index: {VECTOR_INDEX_NAME}...")
        
        vsc.create_delta_sync_index(
            endpoint_name=VECTOR_SEARCH_ENDPOINT,
            index_name=VECTOR_INDEX_NAME,
            source_table_name=source_table,
            pipeline_type="TRIGGERED",  # Sync on-demand (cost-efficient)
            primary_key="chunk_id",
            embedding_source_column="chunk_to_embed",
            embedding_model_endpoint_name=embedding_model,
            columns_to_sync=[  # Additional columns available in search results
                "chunk_id",
                "chunk_to_retrieve",
                "chunk_to_embed",
                "source_file",
                "source_uri",
                "chunk_position"
            ]
        )
        
        # Wait for index to be ready
        print("   Waiting for index provisioning (this may take several minutes)...")
        while True:
            idx = vsc.get_index(
                endpoint_name=VECTOR_SEARCH_ENDPOINT,
                index_name=VECTOR_INDEX_NAME
            )
            status = idx.describe().get("status", {}).get("ready", False)
            if status:
                print(f"✅ Index '{VECTOR_INDEX_NAME}' is READY.")
                break
            detailed_state = idx.describe().get("status", {})
            print(f"   Status: {detailed_state} - waiting...")
            time.sleep(60)
    else:
        raise e

⏳ Creating Delta Sync Index: ashish_kamboj_rag.project_docs_rag.gold_chunks_index...


---------------------------------------------------------------------------
NotFound                                  Traceback (most recent call last)
File <command-8587934572642153>, line 23
     21 try:
     22     # Check if index already exists
---> 23     existing_index = vsc.get_index(
     24         endpoint_name=VECTOR_SEARCH_ENDPOINT,
     25         index_name=VECTOR_INDEX_NAME
     26     )
     27     print(f"✅ Index '{VECTOR_INDEX_NAME}' already exists.")

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-c32540fa-b8f1-4681-9c96-e1eddf27da0a/lib/python3.12/site-packages/databricks/ai_search/client.py:904, in AISearchClient.get_index(self, endpoint_name, index_name)
    896 """
    897 Get an index.
    898 
   (...)
    902 :raises PermissionDenied: If user lacks permission to view the index.
    903 """
--> 904 params = self._fetch_index_params(endpoint_name, index_name)
    905 return AISearchIndex(
    906     workspace_url=self.workspace_url,
    907     index_url=para

In [0]:
# =============================================================================
# TEST: Verify Vector Search is Working
# =============================================================================
# Run a sample similarity search to validate the index is functional.
# =============================================================================

index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=VECTOR_INDEX_NAME
)

# Test query
results = index.similarity_search(
    query_text="What is the Broadcast pipeline architecture?",
    columns=["chunk_to_retrieve", "source_file", "chunk_position"],
    num_results=3
)

print("🔍 Test Query: 'What is the Broadcast pipeline architecture?'")
print("=" * 60)
for i, row in enumerate(results.get("result", {}).get("data_array", []), 1):
    print(f"\n--- Result {i} (Score: {row[-1]:.4f}) ---")
    print(f"Source: {row[1]}")
    print(f"Content: {row[0][:200]}...")

In [0]:
%sql
-- Querying a foundation model service directly in a SQL cell
SELECT ai_query('databricks-llama_3_3_70b_instruct', 'What is prompt engineering?')

---------------------------------------------------------------------------
SparkException                            Traceback (most recent call last)
File <command-5259543193293622>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "-- Querying a foundation model service directly in a SQL cell\nSELECT ai_query('databricks-llama_3_3_70b_instruct', 'What is prompt engineering?')\n")

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:217, 